# Lecture 13: Fine-Tuning Transformers — Answer Key
### NLP Course 2027 — Week 07

---
Complete answers for all four exercises in Lecture 13.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate

MODEL_NAME = 'distilbert-base-uncased'
print(f'Model: {MODEL_NAME}')

---
## Exercise 13.1 — Fine-Tune on IMDb

**Task:** Fine-tune DistilBERT on IMDb (50k reviews). Compare to SST-2 (~91%).

**Key concept:** IMDb is harder than SST-2 — reviews are longer (up to 512 tokens) and contain more nuanced language. DistilBERT typically achieves ~92–93% on IMDb, slightly above SST-2, because the training set is larger.

In [ ]:
def fine_tune_on_dataset(dataset_name, text_col, label_col, num_labels,
                         num_epochs=3, batch_size=32, output_dir='./results'):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    dataset   = load_dataset(dataset_name)

    def tokenize(batch):
        return tokenizer(batch[text_col], truncation=True, max_length=256)

    tokenized = dataset.map(tokenize, batched=True, remove_columns=[text_col])
    if label_col != 'label':
        tokenized = tokenized.rename_column(label_col, 'label')

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    metric = evaluate.load('accuracy')

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return metric.compute(predictions=preds, references=labels)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none',
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['test'],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer.evaluate()

# Fine-tune on IMDb
# (Uncomment to run — takes ~15 min on GPU, ~60 min on CPU)
# results = fine_tune_on_dataset('imdb', 'text', 'label', num_labels=2)
# print(f'IMDb accuracy: {results["eval_accuracy"]:.4f}')
# print('SST-2 baseline: ~0.910 | IMDb expected: ~0.920–0.930')
print('Code ready. Uncomment the last two lines to run fine-tuning.')
print('Expected IMDb accuracy: ~92–93% (larger dataset than SST-2 => slightly better)')

---
## Exercise 13.2 — Multi-Class Fine-Tuning: AG News

**Task:** Fine-tune on AG News (4 categories). Set `num_labels=4`.

**Key concept:** Switching from binary to multi-class classification only requires changing `num_labels`. The Trainer handles multi-class cross-entropy automatically. AG News is a relatively easy 4-class task — expect ~94–95% accuracy.

In [ ]:
label_names = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

# Fine-tune on AG News (uncomment to run)
# results = fine_tune_on_dataset('ag_news', 'text', 'label', num_labels=4,
#                                output_dir='./results/ag_news')
# print(f'AG News accuracy: {results["eval_accuracy"]:.4f}')

# Demonstration: show how to evaluate per-class accuracy
from sklearn.metrics import classification_report

# Simulated results for illustration
np.random.seed(42)
n = 200
true_labels = np.random.randint(0, 4, n)
# Simulate a model that's slightly better than random
pred_labels = np.where(np.random.rand(n) > 0.05, true_labels, (true_labels + 1) % 4)

print('Per-class accuracy (illustration with simulated results):')
print(classification_report(true_labels, pred_labels,
                             target_names=list(label_names.values())))
print('Note: Real DistilBERT on AG News achieves ~94-95% accuracy per class.')

---
## Exercise 13.3 — Early Stopping

**Task:** Add `EarlyStoppingCallback(patience=2)`. How many epochs does training need?

**Key concept:** Early stopping monitors a validation metric and stops when it hasn't improved for `patience` evaluations. This prevents overfitting and saves compute. For DistilBERT on sentiment, training typically converges in 2–4 epochs.

In [ ]:
def fine_tune_with_early_stopping(dataset_name, text_col, num_labels,
                                   max_epochs=10, patience=2):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    dataset   = load_dataset(dataset_name)

    def tokenize(batch):
        return tokenizer(batch[text_col], truncation=True, max_length=256)

    tokenized = dataset.map(tokenize, batched=True, remove_columns=[text_col])
    model  = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    metric = evaluate.load('accuracy')

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        return metric.compute(predictions=np.argmax(logits, -1), references=labels)

    args = TrainingArguments(
        output_dir='./results/early_stopping',
        num_train_epochs=max_epochs,
        per_device_train_batch_size=32,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        greater_is_better=True,
        report_to='none',
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['test'],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)],
    )
    trainer.train()
    return trainer

# trainer = fine_tune_with_early_stopping('imdb', 'text', num_labels=2)
# actual_epochs = trainer.state.epoch
# print(f'Training stopped at epoch {actual_epochs:.0f} (max was 10)')
print('Code ready — uncomment to run.')
print('Expected: training stops at epoch 3–4 on IMDb with patience=2.')
print('Early stopping saves ~6–7 epochs of unnecessary training.')

---
## Exercise 13.4 — Fine-Tuning vs Feature Extraction

**Task:** Compare full fine-tuning (all weights) vs feature extraction (frozen BERT, only head trained).

**Key concept:** Feature extraction is fast but weaker — the pretrained representations may not be optimal for the task. Full fine-tuning adapts all weights but risks overfitting on small datasets. For small labeled datasets (<1k examples), feature extraction often wins.

In [ ]:
def get_model_feature_extraction(num_labels=2):
    """Freeze all BERT weights; only train the classifier head."""
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    for name, param in model.named_parameters():
        if 'classifier' not in name and 'pre_classifier' not in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Feature extraction: {trainable:,} / {total:,} trainable ({100*trainable/total:.2f}%)')
    return model

def get_model_fine_tuning(num_labels=2):
    """All weights trainable."""
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Full fine-tuning:   {trainable:,} trainable (100%)')
    return model

fe_model  = get_model_feature_extraction()
ft_model  = get_model_fine_tuning()

print()
print('Expected results on IMDb:')
print('  Feature extraction: ~85–88% accuracy (frozen representations may not perfectly align)')
print('  Full fine-tuning:   ~92–93% accuracy (all weights adapt to the task)')
print()
print('Rule of thumb:')
print('  < 1,000 labeled examples  -> feature extraction (avoid overfitting)')
print('  > 5,000 labeled examples  -> full fine-tuning (better final accuracy)')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 13.1 | IMDb (50k) typically outperforms SST-2 (67k) — larger data compensates for harder text |
| 13.2 | Multi-class: only `num_labels` changes; Trainer handles CE loss automatically |
| 13.3 | Early stopping with patience=2 typically stops at epoch 3–4, saving 60%+ of compute |
| 13.4 | Feature extraction: fast, ~85%; full fine-tuning: ~93%; choose by dataset size |

---
*NLP Course 2027 — Week 07*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**